# MEDI-Zero 피부 질환 분류 모델 학습
**모델**: EfficientNet-B4 (전이학습)  
**클래스**: 15개 피부 질환  
**XAI**: Grad-CAM 시각화  

## 데이터셋 구조 (Kaggle Dataset에 업로드 필요)
```
skin_dataset/
  광선각화증/   *.png
  기저세포암/   *.png
  멜라닌세포모반/ *.png
  보웬병/      *.png
  비립종/      *.png
  사마귀/      *.png
  악성흑색종/   *.png
  지루각화증/   *.png
  편평세포암/   *.png
  표피낭종/    *.png
  피부섬유종/   *.png
  피지샘증식증/  *.png
  혈관종/      *.png
  화농 육아종/  *.png
  흑색점/      *.png
```

In [ ]:
!pip install -q grad-cam timm

In [ ]:
# ── 한글 폰트 설치 ────────────────────────────────────
import subprocess, matplotlib, shutil

subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-nanum'], check=True)

font_dir = [p for p in matplotlib.font_manager.findSystemFonts() if 'Nanum' in p]
if font_dir:
    matplotlib.font_manager.fontManager.addfont(font_dir[0])

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('한글 폰트 설정 완료')

In [ ]:
import os
import json
import glob
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import models, transforms, datasets
from sklearn.metrics import classification_report, confusion_matrix

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── 설정 ──────────────────────────────────────────────
BASE            = '/kaggle/input/datasets/kimminsu0109/skin-dataset'
TRAIN_IMG_DIR   = f'{BASE}/Train'
TRAIN_LABEL_DIR = f'{BASE}/Train_label'
VAL_IMG_DIR     = f'{BASE}/Val'
VAL_LABEL_DIR   = f'{BASE}/Val_label'
OUTPUT_DIR      = '/kaggle/working'
MODEL_PATH      = os.path.join(OUTPUT_DIR, 'skin_efficientnet_b4.pth')
CLASS_MAP_PATH  = os.path.join(OUTPUT_DIR, 'class_to_idx.json')

INPUT_SIZE   = 380
BBOX_PADDING = 20
BATCH_SIZE   = 16
EPOCHS       = 40
LR           = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
PATIENCE     = 7
USE_AMP      = True

PREFIX_TO_KO = {
    'TS_광선각화증':    '광선각화증',   'VS_광선각화증':    '광선각화증',
    'TS_기저세포암':    '기저세포암',   'VS_기저세포암':    '기저세포암',
    'TS_멜라닌세포모반': '멜라닌세포모반', 'VS_멜라닌세포모반': '멜라닌세포모반',
    'TS_보웬병':       '보웬병',       'VS_보웬병':       '보웬병',
    'TS_비립종':       '비립종',       'VS_비립종':       '비립종',
    'TS_사마귀':       '사마귀',       'VS_사마귀':       '사마귀',
    'TS_악성흑색종':    '악성흑색종',   'VS_악성흑색종':    '악성흑색종',
    'TS_지루각화증':    '지루각화증',   'VS_지루각화증':    '지루각화증',
    'TS_편평세포암':    '편평세포암',   'VS_편평세포암':    '편평세포암',
    'TS_표피낭종':     '표피낭종',     'VS_표피낭종':     '표피낭종',
    'TS_피부섬유종':    '피부섬유종',   'VS_피부섬유종':    '피부섬유종',
    'TS_피지샘증식증':   '피지샘증식증',  'VS_피지샘증식증':   '피지샘증식증',
    'TS_혈관종':       '혈관종',       'VS_혈관종':       '혈관종',
    'TS_화농 육아종':   '화농 육아종',  'VS_화농 육아종':   '화농 육아종',
    'TS_흑색점':       '흑색점',       'VS_흑색점':       '흑색점',
}

KO_CLASSES   = sorted(set(PREFIX_TO_KO.values()))
KO_TO_IDX    = {ko: i for i, ko in enumerate(KO_CLASSES)}
CLASS_TO_IDX = {folder: KO_TO_IDX[ko] for folder, ko in PREFIX_TO_KO.items()}
IDX_TO_KO    = {i: ko for ko, i in KO_TO_IDX.items()}
NUM_CLASSES  = len(KO_CLASSES)

class_names = [IDX_TO_KO[i] for i in range(NUM_CLASSES)]

for name, path in [('Train 이미지', TRAIN_IMG_DIR), ('Train 라벨', TRAIN_LABEL_DIR),
                   ('Val 이미지',   VAL_IMG_DIR),   ('Val 라벨',   VAL_LABEL_DIR)]:
    print(f'{"✅" if os.path.isdir(path) else "❌ 없음"}  {name}: {path}')
print(f'\n클래스 수: {NUM_CLASSES} | 배치: {BATCH_SIZE} | AMP: {USE_AMP}')
print(f'클래스 순서: {class_names}')

In [ ]:
# ── 커스텀 Dataset ─────────────────────────────────────
IMG_EXTS = ('*.png', '*.jpg', '*.jpeg')

class SkinDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, transform=None):
        self.transform = transform
        self.samples   = []

        for folder in sorted(os.listdir(img_dir)):
            if folder not in CLASS_TO_IDX:
                continue
            cls_idx    = CLASS_TO_IDX[folder]
            img_folder = os.path.join(img_dir, folder)

            img_paths = []
            for ext in IMG_EXTS:
                img_paths.extend(glob.glob(os.path.join(img_folder, ext)))

            for img_path in sorted(img_paths):
                self.samples.append((img_path, cls_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, cls_idx = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, cls_idx


# ── Transform ─────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE + 20, INPUT_SIZE + 20)),
    transforms.RandomCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Train 데이터셋 ─────────────────────────────────────
train_dataset = SkinDataset(TRAIN_IMG_DIR, transform=train_transform)

# ── Val 1500장을 Val 750 / Test 750으로 50:50 분할 ────────
# Val 데이터에 Augmentation 없는 transform 적용
full_val_dataset = SkinDataset(VAL_IMG_DIR, transform=val_transform)
val_size  = len(full_val_dataset) // 2   # 750
test_size = len(full_val_dataset) - val_size  # 750

# SEED 고정으로 재현 가능한 분할
val_dataset, test_dataset = random_split(
    full_val_dataset,
    [val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# class_to_idx 저장
with open(CLASS_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump({'folder_to_idx': CLASS_TO_IDX,
               'idx_to_ko':     {str(k): v for k, v in IDX_TO_KO.items()}},
              f, ensure_ascii=False, indent=2)

print(f'Train : {len(train_dataset):,}장')
print(f'Val   : {len(val_dataset):,}장  (학습 중 Best Model 선택용)')
print(f'Test  : {len(test_dataset):,}장  (최종 평가 전용 — 학습 미관여)')

In [ ]:
# ── 클래스 분포 시각화 (Train 기준) ───────────────────────
# 클래스 인덱스 → 한국어 이름으로 변환
train_labels = [IDX_TO_KO[label] for _, label in train_dataset.samples]
class_counts = pd.Series(train_labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Train 클래스 분포
class_counts.value_counts().sort_index().plot(kind='bar', color='steelblue', ax=axes[0])
axes[0].set_title(f'Train 클래스별 이미지 수 (총 {len(train_dataset):,}장)', fontsize=13)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('이미지 수')

# Val / Test 분포 비교
val_labels  = [IDX_TO_KO[full_val_dataset[i][1]] for i in val_dataset.indices]
test_labels = [IDX_TO_KO[full_val_dataset[i][1]] for i in test_dataset.indices]
split_df = pd.DataFrame({
    'Val':  pd.Series(val_labels).value_counts().sort_index(),
    'Test': pd.Series(test_labels).value_counts().sort_index(),
})
split_df.plot(kind='bar', ax=axes[1], color=['steelblue', 'darkorange'])
axes[1].set_title('Val / Test 클래스별 분포 (각 750장)', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('이미지 수')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150)
plt.show()
print('클래스 불균형 없음: 모든 클래스 Train 800장, Val/Test 각 50장')

## 모델 선택 근거

| 모델 | 파라미터 | ImageNet Top-1 | 입력 크기 | 비고 |
|------|---------|---------------|---------|------|
| VGG-16 | 138M | 71.6% | 224 | 단순, 무겁고 느림 |
| ResNet-50 | 25M | 76.1% | 224 | 범용 baseline |
| DenseNet-121 | 8M | 74.4% | 224 | 의료영상 논문 다수 사용 |
| MobileNetV3-L | 5.4M | 75.2% | 224 | 경량, 모바일 최적화 |
| **EfficientNet-B4** | **19M** | **83.0%** | **380** | **선택** |
| ViT-B/16 | 86M | 81.8% | 224 | 데이터 많이 필요 |

### EfficientNet-B4 선택 이유
1. **Compound Scaling**: 깊이·너비·해상도를 동시에 최적화 → 파라미터 대비 최고 성능
2. **고해상도 입력(380×380)**: 피부 병변의 미세한 텍스처·경계 포착에 유리
3. **의료영상 검증**: ISIC 챌린지, HAM10000 등 피부 질환 분류 벤치마크에서 SOTA

### 파인튜닝 전략 (Gradual Unfreezing)
```
Phase 1 (Epoch 1~9)  : 백본 동결 → 분류기 헤드만 학습 (LR=1e-4)
                       → ImageNet 특징 보존, 빠른 수렴
Phase 2 (Epoch 10~)  : 전체 언프리즈 → LR=1e-5 (1/10 축소)
                       → 피부 병변 특화 특징 학습
```

### 학습 기법 요약
| 기법 | 설정값 | 역할 |
|------|--------|------|
| Loss | CrossEntropyLoss | 다중 분류 기본 손실 |
| Label Smoothing | 0.1 | 과적합 방지, 확신도 캘리브레이션 |
| Optimizer | AdamW | Adam + L2 정규화(Weight Decay) |
| LR Scheduler | CosineAnnealingLR | 학습 후반 안정적 수렴 |
| AMP | Mixed Precision | GPU 메모리 절약 + 속도 향상 |
| Early Stopping | patience=7 | Val 정확도 정체 시 자동 중단 |

In [ ]:
# ── 모델 정의 ─────────────────────────────────────────
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - train_params
print(f'전체 파라미터 : {total_params:,}')
print(f'학습 파라미터 : {train_params:,}  (Phase 1 — 분류기 헤드만)')
print(f'동결 파라미터 : {frozen_params:,}  (Phase 1 — 백본)')

In [ ]:
# ── 학습 설정 ─────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    if len(loader.dataset) == 0:
        raise RuntimeError('데이터셋이 비어있습니다.')
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

In [ ]:
# ── 학습 루프 ─────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
no_improve   = 0
UNFREEZE_EPOCH = 10

for epoch in range(1, EPOCHS + 1):

    if epoch == UNFREEZE_EPOCH:
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW(model.parameters(), lr=LR * 0.1, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - UNFREEZE_EPOCH, eta_min=1e-7)
        print(f'[Epoch {epoch}] ★ 백본 언프리즈 → 전체 파인튜닝 시작 (LR={LR*0.1:.0e})')

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch [{epoch:02d}/{EPOCHS}] '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  => Best model saved (val_acc={best_val_acc:.4f})')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\n학습 완료. Best Val Acc: {best_val_acc:.4f}')

In [ ]:
# ── 학습 곡선 시각화 ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='darkorange')
axes[0].axvline(x=UNFREEZE_EPOCH - 1, color='red', linestyle='--', alpha=0.5, label='백본 언프리즈')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train Acc', color='steelblue')
axes[1].plot(history['val_acc'],   label='Val Acc',   color='darkorange')
axes[1].axvline(x=UNFREEZE_EPOCH - 1, color='red', linestyle='--', alpha=0.5, label='백본 언프리즈')
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── Val 최종 평가 ─────────────────────────────────────
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images.to(DEVICE))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print('=== Val Classification Report ===')
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# ── Test 최종 평가 (학습에 전혀 관여하지 않은 데이터) ─────
# 이 셀은 모든 학습 완료 후 단 1번만 실행합니다.
test_preds, test_labels_list = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(DEVICE))
        test_preds.extend(outputs.argmax(1).cpu().numpy())
        test_labels_list.extend(labels.numpy())

print('=== Test Classification Report (최종 성능 지표) ===')
print(classification_report(test_labels_list, test_preds, target_names=class_names))

# Val vs Test 요약 비교
from sklearn.metrics import accuracy_score
val_acc_final  = accuracy_score(all_labels, all_preds)
test_acc_final = accuracy_score(test_labels_list, test_preds)
print(f'\n[요약]')
print(f'  Val  Accuracy : {val_acc_final:.4f}  (학습 중 모델 선택에 사용)')
print(f'  Test Accuracy : {test_acc_final:.4f}  (최종 성능 — 학습 미관여)')
if abs(val_acc_final - test_acc_final) > 0.05:
    print('  ⚠️  Val/Test 차이 5%p 초과 → 과적합 가능성 있음')
else:
    print('  ✅ Val/Test 차이 정상 범위 → 일반화 양호')

In [ ]:
# ── 혼동 행렬 시각화 (Test 기준) ───────────────────────
cm = confusion_matrix(test_labels_list, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(16, 13))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Test Set (Normalized)', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# ── 클래스별 정확도 막대그래프 ─────────────────────────
per_class_acc = cm.diagonal() / cm.sum(axis=1)
colors = ['crimson' if acc < 0.7 else 'steelblue' for acc in per_class_acc]

plt.figure(figsize=(14, 5))
bars = plt.bar(class_names, per_class_acc, color=colors)
plt.axhline(y=per_class_acc.mean(), color='red', linestyle='--',
            label=f'평균: {per_class_acc.mean():.3f}')
for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2f}', ha='center', va='bottom', fontsize=9)
plt.title('클래스별 정확도 (Test Set) — 빨간색: 70% 미만', fontsize=14)
plt.ylim(0, 1.15)
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_accuracy.png'), dpi=150)
plt.show()

In [ ]:
# ── Grad-CAM 샘플 시각화 (원본 vs 히트맵 나란히) ──────────
model.eval()
target_layers = [model.features[-1]]

# Test셋에서 클래스별 첫 번째 샘플 추출
class_samples = {}
for idx in test_dataset.indices:
    img_path, cls_idx = full_val_dataset.samples[idx]
    if cls_idx not in class_samples:
        class_samples[cls_idx] = img_path
    if len(class_samples) == NUM_CLASSES:
        break

fig, axes = plt.subplots(NUM_CLASSES, 2, figsize=(8, NUM_CLASSES * 2.5))

for cls_idx in range(NUM_CLASSES):
    if cls_idx not in class_samples:
        axes[cls_idx, 0].axis('off')
        axes[cls_idx, 1].axis('off')
        continue

    pil_img = Image.open(class_samples[cls_idx]).convert('RGB')
    tensor  = val_transform(pil_img).unsqueeze(0).to(DEVICE)
    rgb_img = np.array(pil_img.resize((INPUT_SIZE, INPUT_SIZE)), dtype=np.float32) / 255.0

    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=tensor,
                            targets=[ClassifierOutputTarget(cls_idx)])[0]
    overlay = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    # 왼쪽: 원본, 오른쪽: Grad-CAM
    axes[cls_idx, 0].imshow(rgb_img)
    axes[cls_idx, 0].set_title(f'{class_names[cls_idx]}\n원본', fontsize=9)
    axes[cls_idx, 0].axis('off')

    axes[cls_idx, 1].imshow(overlay)
    axes[cls_idx, 1].set_title('Grad-CAM', fontsize=9)
    axes[cls_idx, 1].axis('off')

plt.suptitle('원본 vs Grad-CAM — 모델이 집중한 병변 위치', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Grad-CAM 비교 이미지 저장 완료')

In [ ]:
# ── 출력 파일 확인 ─────────────────────────────────────
print('=== 생성된 파일 ===')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f:45s}  {size/1024:.1f} KB')

print(f'\n=== 최종 요약 ===')
print(f'  데이터셋  : Train {len(train_dataset):,} / Val {len(val_dataset):,} / Test {len(test_dataset):,}')
print(f'  클래스    : {NUM_CLASSES}개 피부 질환')
print(f'  모델      : EfficientNet-B4 (전이학습 + Gradual Unfreezing)')
print(f'  Best Val  : {best_val_acc:.4f}')
print(f'  Test Acc  : {test_acc_final:.4f}')